In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Trades Data Collection

In [2]:
trades_df = pd.read_parquet("../data/processed/tradesAInvestALL.parquet")
trades_df

,id,size,reporting_gap,name,trade_type,filing_date,state,trade_date,party,ticker
0,H002482383202608110036,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
1,H002482383202608110038,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
2,H002482383202608110037,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
3,H002482383202608110040,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
4,H002482383202608110042,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
...,...,...,...,...,...,...,...,...,...,...
4417,H002474652202306210013,$50K-$100K,28 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-21,Republican,UNH
4418,H002487714202306210013,$1K-$15K,47 Days,John R. Curtis,buy,2023-08-07,UT,2023-06-21,Republican,UNH
4419,H002474652202306160006,$100K-$250K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH
4420,H002474652202306160007,$50K-$100K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH


In [3]:
# Making ticker name match with yahoo finance data
trades_df["ticker"] = trades_df["ticker"].replace({"BRK.B" : "BRK-B"})

### Stock Data Collection

In [4]:
from pathlib import Path

stock_dir = Path("../data/raw/stock_prices")

stock_files = list(stock_dir.glob("*.parquet"))

print(stock_files)

[PosixPath('../data/raw/stock_prices/AMZN.parquet'), PosixPath('../data/raw/stock_prices/NVDA.parquet'), PosixPath('../data/raw/stock_prices/AAPL.parquet'), PosixPath('../data/raw/stock_prices/META.parquet'), PosixPath('../data/raw/stock_prices/DELL.parquet'), PosixPath('../data/raw/stock_prices/BRK-B.parquet'), PosixPath('../data/raw/stock_prices/SMCI.parquet'), PosixPath('../data/raw/stock_prices/GOOGL.parquet'), PosixPath('../data/raw/stock_prices/MSFT.parquet'), PosixPath('../data/raw/stock_prices/JNJ.parquet'), PosixPath('../data/raw/stock_prices/UNH.parquet')]


In [5]:
stock_data = {}

for file in stock_files:
    ticker = file.stem #.stem takes out .filetype
    stock_data[ticker] = pd.read_parquet(file)

In [6]:
stock_data["AAPL"]

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2016-01-04,23.688675,23.693173,22.935405,23.072567,270597600
2016-01-05,23.095053,23.801103,23.027597,23.778618,223164000
2016-01-06,22.643091,23.018604,22.456461,22.611611,273829600
2016-01-07,21.687445,22.514918,21.682948,22.188876,324377600
2016-01-08,21.802126,22.285569,21.757155,22.159649,283192000
...,...,...,...,...,...
2026-10-01,330.320007,332.480011,325.809998,330.000000,36306300
2026-10-02,333.690002,334.540009,330.609985,333.260010,33278600


In [7]:
trades_df["trade_date"].head()

0   2026-08-11
1   2026-08-11
2   2026-08-11
3   2026-08-11
4   2026-08-11
Name: trade_date, dtype: datetime64[us]

In [8]:
stock_data["AAPL"].index.dtype

dtype('<M8[ms]')

In [9]:
## Making datetime dtype same
trades_df["trade_date"] = pd.to_datetime(trades_df["trade_date"]).astype("datetime64[us]").dt.normalize() # .dt as series .normalize sets any potential time to 00:00:00 (dont care abt time)
trades_df["filing_date"] = pd.to_datetime(trades_df["filing_date"]).astype("datetime64[us]").dt.normalize() # .dt as series .normalize sets any potential time to 00:00:00 (dont care abt time)

for ticker, df in stock_data.items():
    df.index = pd.to_datetime(df.index).astype("datetime64[us]").normalize()

In [10]:
print(trades_df["trade_date"].dtype)
print(trades_df["filing_date"].dtype)
print(stock_data["AAPL"].index.dtype)

datetime64[us]
datetime64[us]
datetime64[us]


In [11]:
aapl = stock_data["AAPL"]

matched = trades_df.loc[trades_df["ticker"] == "AAPL", "trade_date"].isin(aapl.index) # using isin just in case filing date is on a non trading day

In [12]:
for ticker, stock in stock_data.items():
    trades = trades_df.loc[
        trades_df["ticker"] == ticker, "trade_date"
    ]

    matched = trades.isin(stock.index)

    print(
        f"{ticker}: {matched.sum()}/{len(matched)} exact matches" 
    )

AMZN: 525/527 exact matches
NVDA: 493/493 exact matches
AAPL: 578/578 exact matches
META: 437/438 exact matches
DELL: 59/59 exact matches
BRK-B: 345/347 exact matches
SMCI: 26/26 exact matches
GOOGL: 676/678 exact matches
MSFT: 681/684 exact matches
JNJ: 259/259 exact matches
UNH: 332/333 exact matches


In [ ]:
def get_trade_prices(row):
    ticker = row["ticker"]
    trade_date = row["trade_date"]

    stock = stock_data[ticker]

    # Close is currently a DataFrame because of yfinance
    close = stock["Close"][ticker]

    # Find first trading day >= trade_date
    idx = stock.index.searchsorted(trade_date)

    if idx + 30 >= len(stock):
        return pd.Series({
            "price_date": pd.NaT,
            "Close_Trade": pd.NA,
            "Close_1D": pd.NA,
            "Close_5D": pd.NA,
            "Close_10D": pd.NA,
            "Close_30D": pd.NA
        })

    return pd.Series({
        "price_date": stock.index[idx],
        "Close_Trade": close.iloc[idx],
        "Close_1D": close.iloc[idx + 1],
        "Close_5D": close.iloc[idx + 5],
        "Close_10D": close.iloc[idx + 10],
        "Close_30D": close.iloc[idx + 30]
    })
    

In [14]:
prices = trades_df.apply(get_trade_prices, axis=1)

In [15]:
trades_df = pd.concat([trades_df, prices], axis=1)

In [16]:
trades_df[
    [
        "ticker",
        "trade_date",
        "price_date",
        "Close_Trade",
        "Close_1D",
        "Close_5D",
        "Close_10D",
        "Close_30D"
    ]
].head(10)

,ticker,trade_date,price_date,Close_Trade,Close_1D,Close_5D,Close_10D,Close_30D
0,AMZN,2026-08-11,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
1,AMZN,2026-08-11,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
2,AMZN,2026-08-11,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
3,AMZN,2026-08-11,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
4,AMZN,2026-08-11,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
5,AMZN,2026-08-11,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
6,AMZN,2026-08-11,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
7,AMZN,2026-07-08,2026-07-08,243.619995,247.039993,254.960007,244.850006,265.839996
8,AMZN,2026-06-05,2026-06-05,246.029999,245.220001,238.550003,232.789993,247.550003
9,AMZN,2026-06-01,2026-06-01,261.26001,256.519989,245.220001,246.020004,254.960007


In [17]:
print(trades_df["Close_Trade"].isna().sum())
print(trades_df["Close_1D"].isna().sum())
print(trades_df["Close_30D"].isna().sum())

9
9
9


In [21]:
## Will ignore those 9 data points (30 days have not passed since they were filed)
trades_df = trades_df.dropna(subset=["Close_Trade"])

In [23]:
print(trades_df["Close_Trade"].isna().sum())
trades_df.shape

0


(4413, 16)

In [ ]:
trades_df.columns.to_list()

['id',
 'size',
 'reporting_gap',
 'name',
 'trade_type',
 'filing_date',
 'state',
 'trade_date',
 'party',
 'ticker',
 'price_date',
 'Close_Trade',
 'Close_1D',
 'Close_5D',
 'Close_10D',
 'Close_30D']

In [26]:
trades_df.head()

,id,size,reporting_gap,name,trade_type,filing_date,state,trade_date,party,ticker,price_date,Close_Trade,Close_1D,Close_5D,Close_10D,Close_30D
0,H002482383202608110036,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
1,H002482383202608110038,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
2,H002482383202608110037,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
3,H002482383202608110040,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004
4,H002482383202608110042,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004


p.s. won't include fee costs for version 1 of calculating return from longing/shorting

Formula for calculating long returns:
Return (%) = ((Selling price - Purchase Price) / Purchase Price) * 100

Formula for calculating short returns
Return (%) = ((Selling price - Repurchase Cost) / Purchase Price) * 100

https://www.trading.com/blog/cfd/stocks/what-is-long-and-short-position-in-trading

In [27]:
trades_df["position"] = trades_df["trade_type"].map({
    "sell": "short",
    "buy": "long"
})


In [28]:
trades_df

,id,size,reporting_gap,name,trade_type,filing_date,state,trade_date,party,ticker,price_date,Close_Trade,Close_1D,Close_5D,Close_10D,Close_30D,position
0,H002482383202608110036,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004,short
1,H002482383202608110038,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004,short
2,H002482383202608110037,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004,short
3,H002482383202608110040,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004,short
4,H002482383202608110042,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,2026-08-11,272.269989,267.279999,259.450012,261.059998,249.270004,short
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4417,H002474652202306210013,$50K-$100K,28 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-21,Republican,UNH,2023-06-21,445.973328,448.124908,443.821869,439.060425,472.268707,short
4418,H002487714202306210013,$1K-$15K,47 Days,John R. Curtis,buy,2023-08-07,UT,2023-06-21,Republican,UNH,2023-06-21,445.973328,448.124908,443.821869,439.060425,472.268707,long
4419,H002474652202306160006,$100K-$250K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH,2023-06-16,428.892151,439.088501,448.134308,447.030457,472.156464,short
4420,H002474652202306160007,$50K-$100K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH,2023-06-16,428.892151,439.088501,448.134308,447.030457,472.156464,short


In [32]:
trades_df["position"].value_counts()

position
short    2332
long     2081
Name: count, dtype: int64

In [ ]:
trades_df["return"] = np.where(trades_df["position"] == "short", 
                               
                               )

In [33]:
def get_trade_returns(row):
    position = row["position"]
    close_trade = row["Close_Trade"]

    prices = {
        "1D": row["Close_1D"],
        "5D": row["Close_5D"],
        "10D": row["Close_10D"],
        "30D": row["Close_30D"],
    }

    returns = {}

    for period, price in prices.items():
        if position == "long":
            returns[f"return_{period}"] = ((price - close_trade) / close_trade)
        elif position == "short":
            returns[f"return_{period}"] = ((close_trade - price) / price)
        else:
            returns[f"return_{period}"] = np.nan

    return pd.Series(returns)

    
    

In [34]:
returns = trades_df.apply(get_trade_returns, axis=1)

In [35]:
trades_df = pd.concat([trades_df, returns], axis=1)

In [36]:
trades_df

,id,size,reporting_gap,name,trade_type,filing_date,state,trade_date,party,ticker,...,Close_Trade,Close_1D,Close_5D,Close_10D,Close_30D,position,return_1D,return_5D,return_10D,return_30D
0,H002482383202608110036,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,...,272.269989,267.279999,259.450012,261.059998,249.270004,short,0.018670,0.049412,0.042940,0.092269
1,H002482383202608110038,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,...,272.269989,267.279999,259.450012,261.059998,249.270004,short,0.018670,0.049412,0.042940,0.092269
2,H002482383202608110037,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,...,272.269989,267.279999,259.450012,261.059998,249.270004,short,0.018670,0.049412,0.042940,0.092269
3,H002482383202608110040,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,...,272.269989,267.279999,259.450012,261.059998,249.270004,short,0.018670,0.049412,0.042940,0.092269
4,H002482383202608110042,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN,...,272.269989,267.279999,259.450012,261.059998,249.270004,short,0.018670,0.049412,0.042940,0.092269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4417,H002474652202306210013,$50K-$100K,28 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-21,Republican,UNH,...,445.973328,448.124908,443.821869,439.060425,472.268707,short,-0.004801,0.004848,0.015745,-0.055679
4418,H002487714202306210013,$1K-$15K,47 Days,John R. Curtis,buy,2023-08-07,UT,2023-06-21,Republican,UNH,...,445.973328,448.124908,443.821869,439.060425,472.268707,long,0.004824,-0.004824,-0.015501,0.058962
4419,H002474652202306160006,$100K-$250K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH,...,428.892151,439.088501,448.134308,447.030457,472.156464,short,-0.023222,-0.042938,-0.040575,-0.091631
4420,H002474652202306160007,$50K-$100K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH,...,428.892151,439.088501,448.134308,447.030457,472.156464,short,-0.023222,-0.042938,-0.040575,-0.091631


In [38]:
trades_df[["return_1D","return_5D","return_10D","return_30D"]].describe()

,return_1D,return_5D,return_10D,return_30D
count,4413.000000,4413.000000,4413.000000,4413.000000
mean,0.000270,0.003480,0.004299,0.008024
std,0.025355,0.052821,0.074274,0.127557
min,-0.223797,-0.439517,-0.557619,-0.502742
25%,-0.010634,-0.023495,-0.035400,-0.063546
50%,-0.000620,0.002274,0.003230,0.002629
75%,0.010830,0.029750,0.040954,0.069839
max,0.358513,0.409423,0.491641,1.032102
